In [ ]:
%pip install datasets
%pip install biopython
%pip install crewai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import glob
from datasets import load_dataset

# Find all JSONL files in the textbooks/chunk/ directory
jsonl_files = glob.glob("textbooks/chunk/*.jsonl")

# Create data_files dict with all files
data_files = {"train": jsonl_files}

# Load all files into a single dataset
ds = load_dataset("json", data_files=data_files, split="train")

print(f"Total records loaded: {len(ds)}")
print(f"Files loaded: {len(jsonl_files)}")
print(ds[0])


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Total records loaded: 125847
Files loaded: 18
{'id': 'Anatomy_Gray_0', 'title': 'Anatomy_Gray', 'content': 'What is anatomy? Anatomy includes those structures that can be seen grossly (without the aid of magnification) and microscopically (with the aid of magnification). Typically, when used by itself, the term anatomy tends to mean gross or macroscopic anatomy—that is, the study of structures that can be seen without using a microscopic. Microscopic anatomy, also called histology, is the study of cells and tissues using a microscope. Anatomy forms the basis for the practice of medicine. Anatomy leads the physician toward an understanding of a patient’s disease, whether he or she is carrying out a physical examination or using the most advanced imaging techniques. Anatomy is also important for dentists, chiropractors, physical therapists, and all others involved in any aspect of patient treatment that begins with an analysis of clinical signs. The ability to interpret a clinical observ

In [38]:
def parse_title(title: str) -> dict:
    """
    Map an arbitrary textbook chunk title to:
    - textbook_id: normalized textbook identifier, e.g. 'Harrison_Internal_Medicine'
    - specialty: coarse specialty, e.g. 'internal_medicine', 'pharmacology', etc.
    """
    t_low = title.lower()

    textbook_id = None
    specialty = None

    if "anatomy" in t_low:
        textbook_id = "Anatomy_Gray"
        specialty = "anatomy"
    elif "biochemistry" in t_low:
        textbook_id = "Biochemistry_Lippincott"
        specialty = "biochemistry"
    elif "cell" in t_low and "biology" in t_low:
        textbook_id = "Cell_Biology_Alberts"
        specialty = "cell_biology"
    elif "surgery" in t_low or "surgical" in t_low:
        textbook_id = "Surgery_Schwartz"
        specialty = "surgery"
    elif "first aid" in t_low or "step" in t_low:
        textbook_id = "First_Aid_Step1" if "step 1" in t_low else "First_Aid_Step2"
        specialty = "first_aid"
    elif "gynecology" in t_low or "gyneco" in t_low:
        textbook_id = "Gynecology_Novak"
        specialty = "gynecology"
    elif "histology" in t_low:
        textbook_id = "Histology_Ross"
        specialty = "histology"
    elif "immunology" in t_low or "immune" in t_low:
        textbook_id = "Immunology_Janeway"
        specialty = "immunology"
    elif "internal medicine" in t_low or "harriso" in t_low:
        textbook_id = "InternalMed_Harrison"
        specialty = "internal_medicine"
    elif "neurology" in t_low or "neuro" in t_low:
        textbook_id = "Neurology_Adams"
        specialty = "neurology"
    elif "obstetric" in t_low or "obstentric" in t_low:
        textbook_id = "Obstentrics_Williams"
        specialty = "obstetrics"
    elif "pathology" in t_low or "pathoma" in t_low:
        textbook_id = "Pathoma_Husain" if "pathoma" in t_low else "Pathology_Robbins"
        specialty = "pathology"
    elif "pediatric" in t_low or "pediatr" in t_low:
        textbook_id = "Pediatrics_Nelson"
        specialty = "pediatrics"
    elif "pharmacology" in t_low or "pharma" in t_low:
        textbook_id = "Pharmacology_Katzung"
        specialty = "pharmacology"
    elif "physiology" in t_low:
        textbook_id = "Physiology_Levy"
        specialty = "physiology"
    elif "psychiatry" in t_low or "psichiatry" in t_low or "dsm" in t_low:
        textbook_id = "Psichiatry_DSM-5"
        specialty = "psychiatry"

    return {"textbook_id": textbook_id, "specialty": specialty}

In [39]:
def add_metadata(example):
    meta = parse_title(example["title"])
    return {
        "id": example["id"],
        "title": example["title"],
        "text": example["content"],
        **meta,
    }

ds_with_meta = ds.map(add_metadata)


In [40]:
%pip install langchain-community faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
%pip install langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:


from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(
    model_name="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)

texts = [r["text"] for r in ds_with_meta]
""" metadatas = [
    {
        "id": r["id"],
        "title": r["title"],
        "textbook_id": r["textbook_id"],
        "specialty": r["specialty"],
    }
    for r in ds_with_meta
] """

# vectorstore = FAISS.from_texts(texts=texts, embedding=embed_model, metadatas=metadatas)
# vectorstore.save_local("faiss_med_textbooks_jsonl")
vectorstore = FAISS.load_local("faiss_med_textbooks_jsonl", embeddings=embed_model, allow_dangerous_deserialization=True)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from typing import Optional, List, Dict
import json
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

def retrieve_textbook_snippets(
    query: str,
    k: int = 5,
    specialty: Optional[str] = None,
    textbook_id: Optional[str] = None,
) -> List[Dict]:
    filters = {}
    if specialty:
        filters["specialty"] = specialty
    if textbook_id:
        filters["textbook_id"] = textbook_id

    docs = vectorstore.similarity_search(
        query,
        k=k,
        filter=filters if filters else None
    )

    return [
        {
            "text": d.page_content,
            "id": d.metadata.get("id"),
            "title": d.metadata.get("title"),
            "textbook_id": d.metadata.get("textbook_id"),
            "specialty": d.metadata.get("specialty"),
        }
        for d in docs
    ]


In [ ]:
from typing import Annotated, Optional, List, Dict

@tool("search_textbooks")
def search_textbooks(
    query: Annotated[str, "Clinical questions to search in textbooks"],
    k: Annotated[int, "Number of passages to retrieve"] = 5,
    specialty: Annotated[Optional[str], "Optional specialty filter"] = None,
    textbook_id: Annotated[Optional[str], "Optional textbook identifier"] = None,
) -> List[Dict]:
    """
    Search the JSONL textbook corpus for relevant passages.

    Returns items with:
    - text: snippet content
    - id: original record Id
    - title: original Title string
    - textbook_id: normalized identifier (if detected)
    - specialty: coarse specialty (if detected)
    """
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty=specialty,
        textbook_id=textbook_id,
    )

@tool("search_pharmacology")
def search_pharmacology(
    query: Annotated[str, "Question about drugs, doses, interactions, adverse effects"],
    k: int = 5
) -> List[Dict]:
    """Search pharmacology-related textbook content for drug information."""
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty="pharmacology",
    )

@tool("search_radiology")
def search_radiology(
    query: Annotated[str, "Question about imaging findings or which imaging to order"],
    k: int = 5
) -> List[Dict]:
    """Search imaging-related content in textbooks for radiology guidance."""
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty="radiology",
    )

@tool("search_general_medicine")
def search_general_medicine(
    query: Annotated[str, "General medicine question (triage, red flags, basic workup)"],
    k: int = 5
) -> List[Dict]:
    """Search broad, core clinical medicine content in textbooks."""
    return retrieve_textbook_snippets(
        query=query,
        k=k,
        specialty="internal_medicine",  # or use a union of several specialties
    )


In [ ]:
from typing import Annotated, Optional, List, Dict
from Bio import Entrez
import time

# Set up Entrez API email (required by NCBI)
Entrez.email = "zsolt.feher@stud.ki.se"  # Replace with your email

def search_pubmed(
    query: Annotated[str, "Medical/clinical search query for PubMed"],
    k: Annotated[int, "Maximum number of articles to retrieve"] = 5,
    min_year: Annotated[Optional[int], "Optional minimum publication year"] = None,
) -> List[Dict]:
    """
    Search PubMed for relevant articles.
    
    Returns items with:
    - pmid: PubMed ID
    - title: Article title
    - authors: List of authors
    - year: Publication year
    - abstract: Article abstract (if available)
    - journal: Journal name
    """
    try:
        # Build search query
        search_term = query
        if min_year:
            search_term += f" AND {min_year}[PDAT] : 3000[PDAT]"
        
        # Search PubMed (use default XML format for Entrez.read() compatibility)
        handle = Entrez.esearch(db="pubmed", term=search_term, retmax=k)
        search_result = Entrez.read(handle)
        handle.close()
        
        # Access ID list from XML response
        id_list = search_result.get("IdList", [])
        
        if not id_list:
            return [{"note": "No articles found for this query"}]
        
        # Fetch article details
        time.sleep(0.5)  # Rate limiting
        handle = Entrez.efetch(db="pubmed", id=",".join(id_list), rettype="xml")
        fetch_result = Entrez.read(handle)
        handle.close()
        
        articles = []
        for article in fetch_result.get("PubmedArticle", []):
            try:
                med_cite = article.get("MedlineCitation", {})
                article_meta = med_cite.get("Article", {})
                
                # Extract authors
                authors = []
                author_list = article_meta.get("AuthorList", [])
                if author_list:
                    for author in author_list:
                        if "LastName" in author:
                            authors.append(author["LastName"])
                
                # Extract PMID
                pmid = med_cite.get("PMID", "N/A")
                if isinstance(pmid, dict):
                    pmid = pmid.get("#text", "N/A")
                
                # Extract year
                pub_date = article_meta.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
                year = pub_date.get("Year", "N/A")
                
                # Extract abstract
                abstract_section = article_meta.get("Abstract", {})
                abstract_text = "N/A"
                if isinstance(abstract_section, dict):
                    abstract_list = abstract_section.get("AbstractText", [])
                    if abstract_list:
                        if isinstance(abstract_list, list) and len(abstract_list) > 0:
                            abstract_text = str(abstract_list[0])
                        else:
                            abstract_text = str(abstract_list)
                
                articles.append({
                    "pmid": str(pmid),
                    "title": article_meta.get("ArticleTitle", "N/A"),
                    "authors": authors[:3],  # First 3 authors
                    "year": str(year),
                    "journal": article_meta.get("Journal", {}).get("Title", "N/A"),
                    "abstract": abstract_text,  # Full abstract text
                    "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                })
            except Exception as article_error:
                print(f"Error parsing article: {article_error}")
                continue
        
        return articles if articles else [{"note": "Could not parse article details"}]
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return [{"error": f"PubMed search failed: {str(e)}"}]

@tool("search_pubmed_clinical")
def search_pubmed_clinical(
    query: Annotated[str, "Clinical question or case to search in PubMed"],
    k: int = 5
) -> List[Dict]:
    """Search PubMed for clinical evidence and case reports."""
    return search_pubmed(query, k=k, min_year=2019)

@tool("search_pubmed_pharmacology")
def search_pubmed_pharmacology(
    query: Annotated[str, "Drug or medication query for PubMed"],
    k: int = 5
) -> List[Dict]:
    """Search PubMed for pharmacology and drug studies."""
    search_term = f"({query}) AND (pharmacology OR drug OR clinical trial OR adverse effect)"
    return search_pubmed(search_term, k=k, min_year=2018)

In [ ]:
from typing import Annotated, Optional, List, Dict
from pathlib import Path
import pandas as pd

CASE_ROOT = Path(r"Case_1")

def _safe_read_csv(path: Path, nrows: Optional[int] = None) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, nrows=nrows)
        except Exception as e:
            last_error = e
    raise last_error

def _df_to_text(df: pd.DataFrame, max_rows: int = 20) -> str:
    if df.empty:
        return "CSV is empty."

    clipped = df.head(max_rows).fillna("")
    rows_text = []
    for i, row in clipped.iterrows():
        parts = [f"{col}={row[col]}" for col in clipped.columns]
        rows_text.append(f"Row {i+1}: " + "; ".join(parts))
    return "\n".join(rows_text)

@tool("search_case_csvs")
def search_case_csvs(
    user_need: Annotated[str, "What the agent is looking for in the case CSVs"],
    selected_file: Annotated[Optional[str], "Relative or full path of the CSV to open; leave empty to only search"] = None,
    max_candidates: Annotated[int, "Maximum number of candidate CSV files to return"] = 8,
    preview_rows: Annotated[int, "How many rows to preview when converting CSV to text"] = 10,
) -> List[Dict]:
    """
    Search all CSV files under Thesis/Case_1 recursively.
    Mode 1: selected_file is None -> return candidate CSV files for the agent to choose from.
    Mode 2: selected_file is provided -> open that CSV and convert it to text for the conversation.
    """
    if not CASE_ROOT.exists():
        return [{"error": f"Folder not found: {CASE_ROOT.resolve()}"}]

    if selected_file:
        chosen = Path(selected_file)
        

        if not chosen.exists():
            return [{"error": f"Selected file not found: {chosen}"}]

        try:
            df = _safe_read_csv(chosen)
            return [{
                "mode": "opened",
                "file": str(chosen),
                "rows": int(df.shape[0]),
                "columns": list(df.columns),
                "text": _df_to_text(df, max_rows=preview_rows)
            }]
        except Exception as e:
            return [{"error": f"Could not open CSV {chosen}: {str(e)}"}]

    csv_files = list(CASE_ROOT.rglob("*.csv"))
    if not csv_files:
        return [{"error": f"No CSV files found under {CASE_ROOT.resolve()}"}]

    need_tokens = set(user_need.lower().split())
    ranked = []

    for f in csv_files:
        try:
            sample = _safe_read_csv(f, nrows=5)
            cols = [str(c) for c in sample.columns]
            haystack = f"{f.name} {' '.join(cols)}".lower()
            score = sum(1 for t in need_tokens if t in haystack)

            ranked.append({
                "score": score,
                "file": str(f),
                "relative_file": str(f.relative_to(CASE_ROOT)),
                "columns": cols,
                "preview": sample.head(3).fillna("").to_dict(orient="records")
            })
        except Exception as e:
            ranked.append({
                "score": -1,
                "file": str(f),
                "relative_file": str(f.relative_to(CASE_ROOT)),
                "error": str(e)
            })

    ranked.sort(key=lambda x: x["score"], reverse=True)
    return ranked[:max_candidates]

In [ ]:
import requests
from typing import Annotated, Dict, List, Any, Optional

RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"

@tool("search_rxnorm")
def _rxnav_get(path: str, **params) -> Dict[str, Any]:
    resp = requests.get(f"{RXNAV_BASE}/{path.lstrip('/')}", params=params, timeout=20)
    resp.raise_for_status()
    return resp.json()


def _flatten_drug_concepts(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []
    groups = payload.get("drugGroup", {}).get("conceptGroup", []) or []
    for group in groups:
        fallback_tty = group.get("tty")
        for c in group.get("conceptProperties", []) or []:
            rows.append(
                {
                    "rxcui": c.get("rxcui"),
                    "name": c.get("name"),
                    "synonym": c.get("synonym"),
                    "tty": c.get("tty", fallback_tty),
                    "language": c.get("language"),
                    "suppress": c.get("suppress"),
                    "umlscui": c.get("umlscui"),
                }
            )

    seen = set()
    deduped = []
    for row in rows:
        key = (row.get("rxcui"), row.get("tty"), row.get("name"))
        if key not in seen:
            seen.add(key)
            deduped.append(row)
    return deduped


def _get_related_concepts(rxcui: str, keep_tty: Optional[set] = None) -> List[Dict[str, Any]]:
    payload = _rxnav_get(f"rxcui/{rxcui}/allrelated.json")
    groups = payload.get("allRelatedGroup", {}).get("conceptGroup", []) or []

    rows = []
    for group in groups:
        fallback_tty = group.get("tty")
        for c in group.get("conceptProperties", []) or []:
            row = {
                "rxcui": c.get("rxcui"),
                "name": c.get("name"),
                "tty": c.get("tty", fallback_tty),
                "synonym": c.get("synonym"),
            }
            if keep_tty is None or row["tty"] in keep_tty:
                rows.append(row)

    seen = set()
    deduped = []
    for row in rows:
        key = (row.get("rxcui"), row.get("tty"), row.get("name"))
        if key not in seen:
            seen.add(key)
            deduped.append(row)
    return deduped


def _detect_mode(query: str) -> str:
    q = query.strip().replace("-", "")
    if q.isdigit() and len(q) in {10, 11}:
        return "ndc"
    if q.isdigit():
        return "rxcui"
    return "name"


def search_rxnorm(
    query: Annotated[str, "Drug name, NDC, or RxCUI to look up in RxNorm."],
    search_by: Annotated[str, "auto, name, ndc, or rxcui."] = "auto",
    max_results: Annotated[int, "Maximum number of primary results to return."] = 5,
    include_related: Annotated[bool, "Include related brand/generic/dose-form concepts when possible."] = True,
) -> Dict[str, Any]:
    """
    Query RxNorm / RxNav for medication normalization.
    Best for pharmacy review, medication reconciliation, and NDC-to-RxCUI verification.
    """
    mode = _detect_mode(query) if search_by == "auto" else search_by.lower()

    if mode == "name":
        payload = _rxnav_get("drugs.json", name=query)
        candidates = _flatten_drug_concepts(payload)[:max_results]

        result = {
            "query": query,
            "search_by": "name",
            "candidate_count": len(candidates),
            "candidates": candidates,
        }

        if include_related and candidates:
            top_rxcui = candidates[0].get("rxcui")
            if top_rxcui:
                result["related_concepts"] = _get_related_concepts(
                    top_rxcui,
                    keep_tty={"IN", "PIN", "MIN", "BN", "SCD", "SBD", "GPCK", "BPCK"},
                )[:20]

        return result

    if mode == "ndc":
        payload = _rxnav_get("ndcstatus.json", ndc=query.replace("-", ""))
        ndc_status = payload.get("ndcStatus", {}) or {}
        rxcui = ndc_status.get("rxcui")

        result = {
            "query": query,
            "search_by": "ndc",
            "ndc_status": ndc_status,
        }

        if include_related and rxcui:
            result["related_concepts"] = _get_related_concepts(
                rxcui,
                keep_tty={"IN", "PIN", "MIN", "BN", "SCD", "SBD", "GPCK", "BPCK"},
            )[:20]

        return result

    if mode == "rxcui":
        related = _get_related_concepts(
            query,
            keep_tty={"IN", "PIN", "MIN", "BN", "SCD", "SBD", "GPCK", "BPCK"},
        )
        return {
            "query": query,
            "search_by": "rxcui",
            "candidate_count": len(related),
            "related_concepts": related[: max_results if max_results > 0 else 20],
        }

    return {
        "query": query,
        "search_by": mode,
        "error": "search_by must be one of: auto, name, ndc, rxcui",
    }

In [ ]:


clinical_llm = LLM(
    model="openai/gpt-oss-20b",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.2,
)

reasoning_llm = LLM(
    model="medgemma-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.1,
)

# ------------------------------------------------------------------
# 4) Agents in a fixed chain
# ------------------------------------------------------------------

general_physician = Agent(
    role="General Physician",
    goal="Build a structured clinical picture from the case files and propose the primary diagnosis and workup.",
    backstory=(
        "You are a primary care and emergency-oriented physician. "
        "You first inspect the patient case CSV data, then support your reasoning with textbook and PubMed evidence."
    ),
    tools=[
        search_case_csvs,
        search_general_medicine,
        search_textbooks,
        search_pubmed_clinical,
        search_rxnorm,
    ],
    llm=clinical_llm,
    allow_delegation=False,
    verbose=True,
)

radiologist_agent = Agent(
    role="Radiologist",
    goal="Interpret imaging findings and provide an evidence-backed radiology impression and next imaging steps.",
    backstory=(
        "You are a radiologist focused only on imaging interpretation and imaging recommendations."
    ),
    tools=[
        search_radiology,
        search_textbooks,
    ],
    llm=reasoning_llm,
    allow_delegation=False,
    verbose=True,
    multimodal=True,
)

pharmacology_agent = Agent(
    role="Clinical Pharmacology Specialist",
    goal="Review medications for appropriateness, interactions, contraindications, and monitoring.",
    backstory=(
        "You are responsible for medication safety, dose review, drug-disease interactions, and RxNorm-backed normalization."
    ),
    tools=[
        search_case_csvs,
        search_pharmacology,
        search_textbooks,
        search_pubmed_pharmacology,
        search_rxnorm,
    ],
    llm=clinical_llm,
    allow_delegation=False,
    verbose=True,
)

senior_doctor = Agent(
    role="Senior Doctor",
    goal="Synthesize all prior outputs into one final treatment plan.",
    backstory=(
        "You are the supervising physician. You do not explore broadly; you integrate the upstream outputs into a final plan."
    ),
    tools=[
        search_case_csvs,
        search_textbooks,
        search_pubmed_clinical,
    ],
    llm=reasoning_llm,
    allow_delegation=False,
    verbose=True,
)

# ------------------------------------------------------------------
# 5) Sequential tasks
# ------------------------------------------------------------------

task_1_case_review = Task(
    description=(
        "Use the available case CSV tools to identify and open the most relevant patient records.\n"
        "Focus on triage, vitals, ED medication reconciliation, timeline, and key symptoms.\n"
        "Produce a structured case summary with:\n"
        "1. Patient timeline\n"
        "2. Symptoms and vital signs\n"
        "3. Current medications\n"
        "4. Primary working diagnosis\n"
        "5. Differential diagnoses\n"
        "6. Diagnostic workup plan\n"
    ),
    expected_output=(
        "A structured clinical case summary in markdown with explicit sections for timeline, findings, diagnosis, differential, and workup."
    ),
    agent=general_physician,
)

task_2_radiology_review = Task(
    description=(
        "Review the prior case summary and the radiology/image.\n"
        "Use radiology/textbook tools to support interpretation.\n\n"
        "Radiology/image is at 'C:\Users\zsolf\Desktop\Thesis\Case_1\CXR-JPG\s52435728\9b00bfd9-c8d1871a-74623a5d-73f2f0b1-f13aa0ad.jpg'\n\n"
        "Return:\n"
        "1. Imaging impression\n"
        "2. Most likely radiologic explanation\n"
        "3. Differential from imaging perspective\n"
        "4. Recommended next imaging or correlation steps\n"
    ),
    expected_output=(
        "A concise radiology report with impression, differential, and recommended follow-up imaging/correlation."
    ),
    agent=radiologist_agent,
    context=[task_1_case_review],
)

task_3_medication_review = Task(
    description=(
        "Review the case summary and radiology review.\n"
        "Use RxNorm and pharmacology resources to analyze all documented medications.\n\n"
        "Return:\n"
        "1. Medication normalization\n"
        "2. Dose/route/frequency review\n"
        "3. Drug-drug and drug-disease interactions\n"
        "4. Contraindications or cautions\n"
        "5. Monitoring recommendations\n"
        "6. Suggested medication changes if needed\n"
    ),
    expected_output=(
        "A medication safety report in markdown with normalized drugs, risks, and recommendations."
    ),
    agent=pharmacology_agent,
    context=[task_1_case_review, task_2_radiology_review],
)

task_4_final_plan = Task(
    description=(
        "Synthesize the case summary, radiology review, and medication safety review into one final treatment plan.\n\n"
        "Return:\n"
        "1. Final working diagnosis\n"
        "2. Key supporting evidence\n"
        "3. Differential diagnoses\n"
        "4. Treatment plan\n"
        "5. Medication plan\n"
        "6. Diagnostic tests\n"
        "7. Monitoring and reassessment plan\n"
        "8. Consults if needed\n"
    ),
    expected_output=(
        "A final integrated treatment plan in markdown, suitable as the single final answer."
    ),
    agent=senior_doctor,
    context=[task_1_case_review, task_2_radiology_review, task_3_medication_review],
)

# ------------------------------------------------------------------
# 6) Build the sequential crew
# ------------------------------------------------------------------

medical_chain = Crew(
    agents=[
        general_physician,
        radiologist_agent,
        pharmacology_agent,
        senior_doctor,
    ],
    tasks=[
        task_1_case_review,
        task_2_radiology_review,
        task_3_medication_review,
        task_4_final_plan,
    ],
    process=Process.sequential,
    verbose=True,
)

# ------------------------------------------------------------------
# 7) Kickoff
# ------------------------------------------------------------------


result = medical_chain.kickoff()
print(result)